# Bow Valley GOWN data

| code | station_no | GIC_Well_ID | Well_ID | name | group |
|---|---|---|---|---|---|
| 0301 | 05BFG006 | 370217 | 370217 | Marmot Creek Basin S5250 | Marmot |
| 0303 | 05BFG004 | 370209 | 370209 | Marmot Creek Basin N5475 | Marmot |
| 0305 | 05BFG005 | 370221 | 370221 | Marmot Creek Basin N6770 | Marmot |
| 0364 | 05BEG023 | 370213 | 370213 | Many Springs | Canmore |
| 0386 | 05BFG003 | 370218 | 370218 | Marmot Creek Basin N2507E | Marmot |
| 0759 | 05BEG020 | 496371 | 496371 | Exshaw | Canmore |
| 0760 | 05BEG018 | 496372 | 496372 | Canmore Tourist Info | Canmore |
| 0764 | 05BEG021 | 1021710 | **11557173** | Harvie Heights | Canmore |
| 0931 | 05BFG001 | 404368 | 404368 | Evan-Thomas Creek | Marmot |

## Organisation

Data is organised by subject. The nine metadata sheets carry 443 columns
that collapse to 175 distinct value-vectors — the aquifer label alone is stored in 11 columns, the
station code in 12. Each theme table below pulls the related columns together once.

| table | grain | subject |
|---|---|---|
| `well_identity` | well | names, aliases, all the ID systems |
| `well_location` | well | verified coordinates, legal survey, county |
| `well_coordinates` | long | every candidate coordinate **with its source and error** |
| `well_elevations` | long | every candidate elevation **with its source and survey vintage** |
| `well_construction` | well | drilling, casing, liner, seal, diameters, depths |
| `well_completion` | interval | screens, perforations, open/production intervals |
| `well_geology` | well | aquifer, formation, lithology, HSU, surficial + bedrock geology |
| `well_lithology_log` | interval | the driller's depth log, metric with elevations |
| `well_hydraulics` | test | pump tests: static level, rate, drawdown summary |
| `well_hydraulic_readings` | reading | drawdown / recovery time series |
| `well_monitoring` | well | network, status, data streams, ownership, well use |
| `well_assessments` | statement | 1996–2008 hydrograph reviews and recommendations |
| `well_water_quality` / `well_wq_analytes` | sample / analyte | chemistry |
| `well_geophysical_logs` | log | which logs exist |
| `well_area_link` | well | `code → twp_id, huc8` — joins the area tables |
| `area_water_use` / `area_stressors` | township / basin | regional water use and pressures |
| `wl_*` | day | water levels, 2005–2024 daily |
| `ctx_*` | varies | surface water and Lafarge points — **not** these wells |

Supporting tables: **`column_lineage`** records where every one of the 443 source columns went;
**`RAW`** keeps the original per-source frames reachable (`RAW['meta_ags_master']`).

Nothing is written to disk.

In [36]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

ROOT       = Path("/home/py/groundwater")
DATA_DIR   = ROOT / "data"
BOW_VALLEY = DATA_DIR / "bow_valley"
GOWN_DIR   = DATA_DIR / "gown"
XLX        = GOWN_DIR / "xlx"
AGS        = BOW_VALLEY / "gown" / "AGS"
AWWID      = BOW_VALLEY / "awwid"
XSEC       = BOW_VALLEY / "cross_section"
STREAM     = BOW_VALLEY / "stream"
BIGHORN    = BOW_VALLEY / "bighorn"

FT = 0.3048     # AWWID stores depths and elevations in feet
IN = 0.0254     # ... and diameters / slot sizes in inches

In [37]:
CODES = ["0301", "0303", "0305", "0364", "0386", "0759", "0760", "0764", "0931"]

ids = pd.DataFrame(
    [
        ("0301", "05BFG006",  301,          370217,       370217,  "Marmot Creek Basin S5250",  "Marmot"),
        ("0303", "05BFG004",  303,          370209,       370209,  "Marmot Creek Basin N5475",  "Marmot"),
        ("0305", "05BFG005",  305,          370221,       370221,  "Marmot Creek Basin N6770",  "Marmot"),
        ("0364", "05BEG023",  364,          370213,       370213,  "Many Springs",              "Canmore"),
        ("0386", "05BFG003",  386,          370218,       370218,  "Marmot Creek Basin N2507E", "Marmot"),
        ("0759", "05BEG020",  759,          496371,       496371,  "Exshaw",                    "Canmore"),
        ("0760", "05BEG018",  760,          496372,       496372,  "Canmore Tourist Info",      "Canmore"),
        ("0764", "05BEG021",  764,         1021710,     11557173,  "Harvie Heights",            "Canmore"),
        ("0931", "05BFG001",  931,          404368,       404368,  "Evan-Thomas Creek",         "Marmot"),
    ],
    columns=["code", "station_no", "station_code", "gic_well_id", "well_id", "short_name", "group"],
).set_index("code").reindex(CODES)

MARMOT  = ["0931", "0301", "0303", "0386", "0305"]
CANMORE = ["0764", "0760", "0759", "0364"]

CODE_BY_STATION_NO = {v: k for k, v in ids["station_no"].items()}
CODE_BY_GIC        = {v: k for k, v in ids["gic_well_id"].items()}

ids

,station_no,station_code,gic_well_id,well_id,short_name,group
code,,,,,,
0301,05BFG006,301,370217,370217,Marmot Creek Basin S5250,Marmot
0303,05BFG004,303,370209,370209,Marmot Creek Basin N5475,Marmot
0305,05BFG005,305,370221,370221,Marmot Creek Basin N6770,Marmot
0364,05BEG023,364,370213,370213,Many Springs,Canmore
0386,05BFG003,386,370218,370218,Marmot Creek Basin N2507E,Marmot
0759,05BEG020,759,496371,496371,Exshaw,Canmore
0760,05BEG018,760,496372,496372,Canmore Tourist Info,Canmore
0764,05BEG021,764,1021710,11557173,Harvie Heights,Canmore
0931,05BFG001,931,404368,404368,Evan-Thomas Creek,Marmot


In [38]:
def to_code(values, kind):
    """Map a source's own well key onto the 4-char zero-padded `code`."""
    s = pd.Series(values)
    if kind == "station_no":
        return s.map(CODE_BY_STATION_NO)
    if kind == "gic":
        return pd.to_numeric(s, errors="coerce").map(CODE_BY_GIC)
    n = pd.to_numeric(s, errors="coerce")
    return n.map(lambda v: f"{int(v):04d}" if pd.notna(v) else None)


def select_wells(df, key, kind="station_code", unique=False):
    """Filter `df` to the 9 wells and put `code` first (or make it the index)."""
    d = df.copy()
    d.insert(0, "code", to_code(d[key], kind).values)
    d = d[d["code"].isin(CODES)]
    if unique:
        assert not d["code"].duplicated().any(), f"expected one row per well via {key}"
        return d.set_index("code").reindex(CODES)
    return d.sort_values("code").reset_index(drop=True)


def add_metric(df, ft_cols=(), in_cols=()):
    """Append metric twins of imperial columns, leaving the originals untouched."""
    d = df.copy()
    for c in ft_cols:
        if c in d.columns:
            d[f"{c}_m"] = pd.to_numeric(d[c], errors="coerce") * FT
    for c in in_cols:
        if c in d.columns:
            d[f"{c}_m"] = pd.to_numeric(d[c], errors="coerce") * IN
    return d

## 1 · Read the sources into `RAW`

Every source frame is loaded once and parked in `RAW`. Nothing downstream reads a file directly, so
`column_lineage` can account for every column that entered the notebook.

In [39]:
RAW = {}

# --- gown/xlx/04 : EPA working-set station info -------------------------------
RAW["meta_station_info"] = select_wells(
    pd.read_csv(XLX / "04_gown_station_info.csv"), "Obs. Well #", unique=True)

# --- gown/xlx/01 : provincial inventory + GIS intersections -------------------
_x01 = pd.ExcelFile(XLX / "01_EPA_GOWN_Intersections.xlsx")
RAW["meta_epa_intersections"] = select_wells(
    _x01.parse("GOWN Intersections"), "GOWN_NO", unique=True)
ref_layer_legend = _x01.parse("Data Summary")

# --- gown/xlx/03 : formation-harmonisation workbook ---------------------------
_x03 = pd.ExcelFile(XLX / "03_GOWN_Formations_Merged.xlsx")
RAW["meta_formation_list"] = select_wells(
    _x03.parse("Well formation list"), "Station Code Gown", unique=True)
RAW["meta_formation_worksheet"] = select_wells(
    _x03.parse("Worksheet"), "Station Code Gown", unique=True)

# 'Metadata Compiled' has a two-row header: sheet row 0 = machine names, row 1 = human labels.
_compiled = _x03.parse("Metadata Compiled")
_labels = _compiled.iloc[0]
_compiled = _compiled.iloc[1:].reset_index(drop=True)
_compiled.columns = [
    str(lbl).replace("\n", " ").strip()
    if (str(c).startswith("Unnamed") or str(c).isdigit()) and pd.notna(lbl) else str(c)
    for c, lbl in zip(_compiled.columns, _labels)
]
RAW["meta_compiled"] = select_wells(_compiled, "stnstationcodegown", unique=True)

# The Mar-2025 allocation review covered mostly Athabasca oil-sands wells; header sits on row 1.
meta_allocations = select_wells(_x03.parse("GOWN well allocations - Mar2025", header=1),
                                "Station Code Gown")

# --- bow_valley/gown/AGS : the pre-merged master ------------------------------
RAW["meta_ags_master"] = select_wells(
    pd.read_csv(AGS / "gown_master_dataset.csv", low_memory=False), "gown_no", unique=True)

# --- gown/tables : live WISKI station metadata --------------------------------
RAW["meta_wiski"] = select_wells(
    pd.read_csv(GOWN_DIR / "tables" / "stations_meta.csv", low_memory=False),
    "groundwater.StationCode_S", unique=True)
RAW["meta_wiski_stations"] = select_wells(
    pd.read_csv(GOWN_DIR / "tables" / "stations.csv"), "gown", unique=True)

# --- derived cross-section collars (metric, DEM-sampled) ----------------------
_collars = pd.read_csv(XSEC / "collars.csv")
_collars["code"] = _collars["code"].astype(str).str.zfill(4)
RAW["xs_collars"] = _collars.set_index("code").reindex(CODES)

# Lookup tables (not well-keyed, so outside the lineage accounting).
ref_ab_formations  = pd.read_excel(XLX / "02_Alberta_formations.xlsx", "alberta_formations_sorted_sjb")
ref_ags_formations = pd.read_csv(AGS / "ref_alberta_formations_lookup.csv")

_n_cols = sum(len(d.columns) for d in RAW.values())
print(f"{len(RAW)} source frames, {_n_cols} columns total")
for k, v in RAW.items():
    print(f"   {k:<26} {str(v.shape):>10}")
print(f"\nmeta_allocations rows for our wells: {len(meta_allocations)} "
      f"(AGS in_allocations: {RAW['meta_ags_master']['in_allocations'].unique()})")

9 source frames, 443 columns total
   meta_station_info             (9, 19)
   meta_epa_intersections       (9, 137)
   meta_formation_list            (9, 4)
   meta_formation_worksheet      (9, 17)
   meta_compiled                 (9, 65)
   meta_ags_master              (9, 112)
   meta_wiski                    (9, 41)
   meta_wiski_stations           (9, 13)
   xs_collars                    (9, 35)

meta_allocations rows for our wells: 0 (AGS in_allocations: [False])


## 2 · The theme builder

`THEMES` is a declarative map: for each theme, each target column lists its candidate sources in
priority order (**WISKI → AGS → EPA → compiled → xlx04 → xs**). The builder takes the first non-null
value, and at the same time emits a `column_lineage` row for every source column it touched, tagged:

| status | meaning |
|---|---|
| `primary` | first source listed; its values won |
| `duplicate` | identical to the target across all 9 wells — a redundant copy |
| `fallback` | supplied values the higher-priority sources were missing |
| `conflict` | disagrees with the target where both are populated |

Because the lineage is emitted by the same pass that builds the tables, it cannot drift out of sync.

In [40]:
def _norm_vec(s):
    """Normalise a 9-well series for comparison (numbers, dates and text alike)."""
    out = []
    for v in s.reindex(CODES):
        if pd.isna(v):
            out.append("~")
            continue
        t = str(v).strip()
        try:
            out.append(f"{float(t):.6g}")
            continue
        except ValueError:
            pass
        d = pd.to_datetime(t, errors="coerce")
        if pd.notna(d) and any(ch in t for ch in "-/"):
            out.append(d.date().isoformat())
            continue
        out.append(" ".join(t.split()).casefold())
    return tuple(out)


LINEAGE = []


def build_theme(theme, spec):
    """spec: [(target_col, [(frame_name, source_col), ...]), ...] in priority order."""
    out = pd.DataFrame(index=pd.Index(CODES, name="code"))
    for target, sources in spec:
        col = pd.Series(np.nan, index=CODES, dtype=object)
        for frame_name, src_col in sources:
            f = RAW[frame_name]
            assert src_col in f.columns, f"{frame_name}.{src_col} not found"
            col = col.where(col.notna(), f[src_col].reindex(CODES))
        out[target] = col
        # Classify each contributing column against the result.
        tgt_vec = _norm_vec(out[target])
        for rank, (frame_name, src_col) in enumerate(sources):
            s = RAW[frame_name][src_col]
            svec = _norm_vec(s)
            if rank == 0:
                status = "primary"
            elif svec == tgt_vec:
                status = "duplicate"
            elif all(a == b for a, b in zip(svec, tgt_vec) if a != "~"):
                status = "fallback"
            else:
                status = "conflict"
            LINEAGE.append({"source_frame": frame_name, "source_column": src_col,
                            "theme": theme, "target_column": target, "status": status,
                            "n_populated": int(s.notna().sum())})
    return out


def note_columns(pairs, status, theme="", target=""):
    """Record columns that do not feed a theme table (all-null, dropped, area-keyed, ...)."""
    for frame_name, src_col in pairs:
        LINEAGE.append({"source_frame": frame_name, "source_column": src_col, "theme": theme,
                        "target_column": target, "status": status,
                        "n_populated": int(RAW[frame_name][src_col].notna().sum())})

### 2.1 Identity, location and monitoring

In [41]:
W, A, E, K, X, S = ("meta_wiski", "meta_ags_master", "meta_epa_intersections",
                    "meta_compiled", "xs_collars", "meta_station_info")
WS, FL, FW = "meta_wiski_stations", "meta_formation_list", "meta_formation_worksheet"

well_identity = build_theme("well_identity", [
    ("station_no",        [(W, "station_no"), (WS, "station_no"), (E, "GOWN_WISKI_StationNumber"),
                           (A, "wiski_station_no"), (K, "stnnumber"), (S, "Station Number"),
                           (X, "station_no")]),
    ("station_code",      [(W, "groundwater.StationCode_S"), (WS, "gown"), (E, "GOWN_NO"),
                           (A, "gown_no"), (K, "stnstationcodegown"), (K, "stnstationcodegown.1"),
                           (K, "stnstationcodegown.2"), (K, "stnstationcodegown.3"),
                           (FL, "Station Code Gown"), (FW, "Station Code Gown"),
                           (S, "Obs. Well #"), (X, "station_code")]),
    ("station_name",      [(W, "station_name"), (WS, "station_name"), (E, "GOWN_STATION_NAME"),
                           (A, "station_name"), (A, "station_name_epa"), (K, "stnname"),
                           (S, "Station Name")]),
    ("name_provincial",   [(E, "GOWN_HistoricalProvincialStationWellName"),
                           (K, "stnprovincialstationwellname")]),
    ("name_assessment_db", [(E, "GOWN_HistoricalAssessmentDatabaseWellName"),
                            (K, "stnassessmentdatabasewellname")]),
    ("aliases",           [(K, "stnmiscstationaliases")]),
    ("alias_remarks",     [(K, "stnmiscstationaliasesremarks")]),
    ("gic_well_id",       [(W, "groundwater.GICWellID_S"), (E, "GOWN_AWWID_ID"), (A, "awwid_id"),
                           (K, "stngicwellid"), (S, "AWWID Well ID"), (X, "gic_well_id")]),
    ("well_id_awwid",     [(X, "well_id")]),
    ("wds_station_no",    [(E, "GOWN_WDS_StationNo"), (K, "stnwdsstationno")]),
    ("wiski_station_id",  [(W, "station_id"), (WS, "station_id")]),
    ("portal_url",        [(WS, "url")]),
    ("object_type",       [(W, "object_type")]),
    ("site_no",           [(W, "site_no")]),
    ("site_name",         [(W, "site_name")]),
])
well_identity.insert(0, "short_name", ids["short_name"])
well_identity.insert(1, "group", ids["group"])

well_location = build_theme("well_location", [
    ("latitude",          [(W, "station_latitude"), (WS, "latitude"), (E, "GOWN_Latitude"),
                           (A, "latitude"), (K, "stnlatitude"), (S, "Latitude"), (X, "latitude")]),
    ("longitude",         [(W, "station_longitude"), (WS, "longitude"), (E, "GOWN_Longitude"),
                           (A, "longitude"), (K, "stnlongitude"), (S, "Longitude"),
                           (X, "longitude")]),
    ("x_utm11n",          [(X, "x_utm11n")]),
    ("y_utm11n",          [(X, "y_utm11n")]),
    ("x_10tm",            [(X, "x_10tm")]),
    ("y_10tm",            [(X, "y_10tm")]),
    ("utm_zone",          [(E, "GOWN_UTMZone"), (A, "utm_zone"), (K, "stnutmzone")]),
    ("latlon_quality",    [(E, "GOWN_LatLongQuality"), (A, "latlon_quality"),
                           (K, "stnlatlongquality")]),
    ("lsd",               [(W, "general.LSD_S"), (E, "GOWN_LSD"), (K, "stnlsd"), (X, "lsd")]),
    ("section",           [(W, "general.Section_S"), (E, "GOWN_SEC"), (K, "stnsection"),
                           (X, "section")]),
    ("township",          [(W, "general.Township_S"), (E, "GOWN_TWP"), (K, "stntownship"),
                           (X, "township")]),
    ("range",             [(W, "general.Range_S"), (E, "GOWN_RGE"), (K, "stnrange"), (X, "range")]),
    ("meridian",          [(W, "general.Meridian_S"), (E, "GOWN_Mer"), (K, "stnmeridian"),
                           (X, "meridian")]),
    ("ats_description",   [(S, "ATS Description")]),
    ("county",            [(E, "GOWN_County"), (A, "county"), (K, "stnisin")]),
    ("map_sheet",         [(E, "GOWN_MapSheet"), (K, "stnmapsheet")]),
    ("location_note",     [(E, "GOWN_LocationDescription"), (K, "stnlocationdescription")]),
])
for _c in ["latitude", "longitude", "x_utm11n", "y_utm11n", "x_10tm", "y_10tm"]:
    well_location[_c] = pd.to_numeric(well_location[_c])

well_monitoring = build_theme("well_monitoring", [
    ("status",            [(A, "status"), (E, "GOWN_STATUS"), (K, "stnstatus")]),
    ("status_detail",     [(A, "status_detail"), (E, "GOWN_OperationalStatusDetails")]),
    ("status_aug2024",    [(A, "status_aug2024"),
                           (K, "GOWN MONITORING Status Aug 2024 from Groundwater _Obs-wells LANDOWNER**")]),
    ("asset_status",      [(K, "WELL SITE PROPERTY ASSET STATUS")]),
    ("last_status",       [(W, "last_status")]),
    ("last_status_date",  [(W, "last_status_date")]),
    ("status_history",    [(W, "diary")]),
    ("status_history_html", [(W, "station_diary_status")]),
    ("wl_active",         [(W, "wl_active")]),
    ("terminal",          [(W, "terminal")]),
    ("wiski_category",    [(WS, "category")]),
    ("static_wl_available", [(A, "static_wl_available"), (K, "stnwl")]),
    ("primary_network",   [(A, "primary_network"), (E, "GOWN_PrimaryWellNetwork")]),
    ("sub_network",       [(A, "sub_network"), (E, "GOWN_SubWellNetwork")]),
    ("nrt_station",       [(A, "nrt_station"), (E, "GOWN_NRT_STN")]),
    ("drought_network",   [(A, "drought_network"), (E, "GOWN_Drought")]),
    ("data_streams",      [(A, "data_streams"), (S, "Data Type")]),
    ("web_data_type",     [(W, "groundwater.WebGWDataType"), (WS, "data_type")]),
    ("wiski_template",    [(W, "groundwater.WISKIStatTemplate_K"), (WS, "gw_type"),
                           (E, "GOWN_WISKI_StationTemplate")]),
    ("wq_class",          [(WS, "water_quality")]),
    ("in_use_since",      [(A, "in_use_since"), (E, "GOWN_InUseSince"), (K, "stninusesince")]),
    ("end_of_data_note",  [(A, "end_of_data_note"), (E, "GOWN_EndOfAvailableData"),
                           (K, "stnendofavailabledata")]),
    ("equip_install_date", [(A, "equip_install_date"), (S, "Equipment Installation Date")]),
    ("station_type",      [(E, "GOWN_AEP_StatIonType"), (K, "stnaenvstationtype")]),
    ("well_use",          [(X, "well_use")]),
    ("owner",             [(A, "owner"), (E, "GOWN_Owner"), (K, "Well Owner"), (K, "stnowner")]),
    ("land_owner",        [(E, "GOWN_LandOwner")]),
    ("monitoring_branch", [(A, "monitoring_branch"), (E, "GOWN_MonitoringBranch"),
                           (K, "stnaemonitoringbranch")]),
    ("aep_region",        [(E, "GOWN_ANEV_Region")]),
    ("contact_name",      [(E, "GOWN_ContactName")]),
    ("in_working_set",    [(A, "in_working_set")]),
    ("in_formation_list", [(A, "in_formation_list")]),
    ("in_allocations",    [(A, "in_allocations")]),
])

for _n in ["well_identity", "well_location", "well_monitoring"]:
    print(f"{_n:<18} {str(eval(_n).shape):>9}")
well_identity[["short_name", "station_no", "station_code", "gic_well_id", "wds_station_no", "aliases"]]

well_identity        (9, 17)
well_location        (9, 17)
well_monitoring      (9, 33)


,short_name,station_no,station_code,gic_well_id,wds_station_no,aliases
code,,,,,,
0301,Marmot Creek Basin S5250,05BFG006,301,370217.0,ABG0370217,WEPA #383 / S250 WT/ RM38
0303,Marmot Creek Basin N5475,05BFG004,303,370209.0,ABG0370209,WEPA #382 / 4575 WTSR20
0305,Marmot Creek Basin N6770,05BFG005,305,370221.0,ABG0370221,WEPA #381 / Thesis= #6770 / GIC= Marmot Creek ...
0364,Many Springs,05BEG023,364,370213.0,ABG0370213,NaN
0386,Marmot Creek Basin N2507E,05BFG003,386,370218.0,ABG0370218,WEPA #379
0759,Exshaw,05BEG020,759,496371.0,NaN,WEPA_#427
0760,Canmore Tourist Info,05BEG018,760,496372.0,NaN,WEPA_#428 or 447?
0764,Harvie Heights,05BEG021,764,1021710.0,NaN,NaN
0931,Evan-Thomas Creek,05BFG001,931,404368.0,NaN,WEPA_#457


### 2.2 Geology, construction and the area link

In [42]:
well_geology = build_theme("well_geology", [
    ("aquifer",           [(W, "groundwater.Aquifer_S"), (A, "aquifer_updated"), (A, "hsu_1"),
                           (E, "GOWN_WISKI_Aquifer"), (E, "GOWN_Aquifer_Updated"), (E, "GOWN_HSU_1"),
                           (K, "stnaquifer"), (K, "Aquifer (Updated)"), (K, "HSU.1"),
                           (S, "Aquifer Name"), (X, "aquifer")]),
    ("hsu_2",             [(A, "hsu_2"), (E, "GOWN_HSU_2"), (K, "HSU.2")]),
    ("lithology",         [(W, "groundwater.Lithology_S"), (A, "lithology"), (E, "GOWN_Lithology"),
                           (K, "stnlithology"), (S, "Lithology"), (X, "aquifer_lithology")]),
    ("aquifer_type",      [(W, "groundwater.AquiferType_K"), (S, "Aquifer Type"),
                           (X, "aquifer_type")]),
    ("aquifer_type_clean", [(A, "aquifer_type_clean"), (A, "aquifer_type_raw"),
                            (E, "GOWN_AquiferType")]),
    ("formation_coarse",  [(A, "formation_coarse"), (E, "GOWN_WISKI_Formation"),
                           (E, "GOWN_Formation_Updated"), (K, "stnformation"),
                           (K, "Formation (Updated)")]),
    ("fm_group",          [(A, "fm_group"), (A, "fm_group_raw"), (FL, "Formation"),
                           (FW, "Converted to Formation"), (FW, "Merged")]),
    ("fm_group_source",   [(A, "fm_group_source")]),
    ("strat_group_id",    [(A, "strat_group_id")]),
    ("strat_sort_key",    [(A, "strat_sort_key")]),
    ("litho_class",       [(A, "litho_class")]),
    ("litho_class_source", [(A, "litho_class_source")]),
    ("quat_subdiv",       [(A, "quat_subdiv"), (FL, "Quaternary subdivisions"),
                           (FW, "Quaternary subdivisions")]),
    ("quat_unit",         [(A, "quat_unit"), (A, "quat_unit_raw"), (FL, "Quaternary_L2"),
                           (FW, "Quaternary_L2")]),
    ("screened_channel",  [(A, "screened_channel_flag")]),
    ("ags_previously_assigned", [(A, "ags_previously_assigned"), (K, "AGS Previously Assigned"),
                                 (FW, "AGS Previously Assigned")]),
    ("geol_update_source", [(A, "geol_update_source"), (E, "GOWN_UpdateSource"),
                            (K, "Update Source")]),
    ("depth_class",       [(A, "depth_class"), (E, "GOWN_DepthClass")]),
    ("areal_extent",      [(A, "areal_extent"), (E, "GOWN_ArealExtent")]),
    ("surficial_geology", [(A, "surficial_geology"), (E, "AGS_SurficialGeologyGeneralized")]),
    ("bedrock_subcrop",   [(A, "bedrock_subcrop"), (E, "AGS_BedrockGeology")]),
    ("ags_surf_aq_type",  [(A, "ags_surf_aq_type"), (E, "AGS_AquiferHostingSediments_AQ_type")]),
    ("ags_surf_aq_conf",  [(A, "ags_surf_aq_conf"), (E, "AGS_AquiferHostingSediments_AQ_conf")]),
    ("aer_base_protection_fm", [(A, "aer_base_protection_fm"),
                                (E, "AER_BaseGroundwaterProtectionFormation")]),
])

well_construction = build_theme("well_construction", [
    ("drill_date",        [(W, "groundwater.DrillDate_D"), (A, "drill_date"),
                           (E, "GOWN_WellDrillDate"), (K, "stndrilldate"), (S, "Drill Date"),
                           (X, "drill_date")]),
    ("well_depth_m",      [(A, "well_depth_m"), (E, "GOWN_WellDepth_m"), (K, "stndepthm"),
                           (S, "Well Depth (m)"), (X, "well_depth_m")]),
    ("depth_str",         [(W, "groundwater.Depth_N"), (WS, "depth"), (S, "Well Depth (mbtoc)"),
                           (X, "depth_str")]),
    ("td_m",              [(X, "td_m")]),
    ("well_diameter_mm",  [(A, "well_diameter_mm"), (E, "GOWN_WellDiameter_mm"),
                           (K, "stndiametermm")]),
    ("completion_type",   [(A, "completion_type"), (E, "GOWN_Completion"), (K, "stncompletion")]),
    ("type_of_work",      [(X, "type_of_work")]),
    ("drilling_method",   [(X, "drilling_method")]),
    ("casing_material",   [(X, "casing_material")]),
    ("casing_od_in",      [(X, "casing_od_in")]),
    ("casing_bottom_m",   [(X, "casing_bottom_m")]),
])
for _c in ["well_depth_m", "td_m", "well_diameter_mm", "casing_od_in", "casing_bottom_m"]:
    well_construction[_c] = pd.to_numeric(well_construction[_c])

well_area_link = build_theme("well_area_link", [
    ("twp_id",            [(A, "twp_id"), (E, "QT1_Combined_GW_Use_TWP_Diversion_Descriptor"),
                           (E, "QT1_Combined_GW_Use_TWP_Return_Descriptor"),
                           (E, "QT1_Combined_SW_Use_TWP_Diversion_Descriptor"),
                           (E, "QT1_Combined_SW_Use_TWP_Return_Descriptor")]),
    ("huc8",              [(A, "huc8"), (E, "HUC_8")]),
    ("huc8_name",         [(A, "huc8_name"), (E, "HUC_8_NAME")]),
    ("huc8_basin",        [(A, "huc8_basin"), (E, "HUC_8_BASIN")]),
    ("huc6",              [(A, "huc6"), (E, "HUC_6")]),
    ("huc6_name",         [(A, "huc6_name"), (E, "HUC_6_NAME")]),
    ("drainage_basin",    [(W, "general.DrainageBasin_S"), (A, "drainage_basin"),
                           (A, "river_basin_epa"), (E, "GOWN_DrainageBasin"),
                           (K, "stndrainagebasin"), (S, "River Basin")]),
    ("sub_basin",         [(A, "sub_basin"), (E, "GOWN_SubBasin"), (K, "stnsubbasin")]),
    ("catchment",         [(W, "catchment_name"), (WS, "catchment")]),
])

for _n in ["well_geology", "well_construction", "well_area_link"]:
    print(f"{_n:<20} {str(eval(_n).shape):>9}")
well_geology[["aquifer", "lithology", "aquifer_type", "formation_coarse", "quat_unit",
              "litho_class", "depth_class", "surficial_geology"]]

well_geology           (9, 24)
well_construction      (9, 11)
well_area_link          (9, 9)


,aquifer,lithology,aquifer_type,formation_coarse,quat_unit,litho_class,depth_class,surficial_geology
code,,,,,,,,
0301,Rocky Mountain,Sandstone,---,Bedrock,NaN,NaN,Shallow,Moraine
0303,Rocky Mountain,Sandstone,---,Bedrock,NaN,NaN,Intermediate,Moraine
0305,Fernie,Shale,---,Bedrock,NaN,shale-dominated,Shallow,Colluvial Deposits
0364,Buried Valley,Gravel,Confined,Channel,Buried Valley,unconsolidated,Intermediate,Fluvial Deposits
0386,Surficial,Gravel & Clay,---,Surficial,Surficial,unconsolidated,Shallow,Fluvial Deposits
0759,Calgary Valley,Clay & Gravel,Confined,Channel,Calgary Valley,unconsolidated,Deep,Fluvial Deposits
0760,Calgary Valley,Clay & Gravel,Unconfined,Channel,Calgary Valley,unconsolidated,Intermediate,Fluvial Deposits
0764,Surficial,Sand & Gravel,---,Surficial,Surficial,unconsolidated,Intermediate,Moraine
0931,Surficial,Gravel,Unconfined,Surficial,Surficial,unconsolidated,Intermediate,Fluvial Deposits


## 3 · Elevations and coordinates — kept sourced, not collapsed

These are the two places where the sources genuinely disagree, so instead of picking a winner each
one gets a long table naming every candidate value and its origin.

**Elevations.** Up to four different top-of-casing elevations exist per well. `GOWN_ElevationQuality`
shows why: the AGS/EPA value is a **2019 RTN GNSS CGVD2013 ±5 cm** re-survey, while WISKI's
`GWREF_DATUM` is the older datum. Section 4 proves the published water levels are still reduced to
`GWREF_DATUM` — the re-survey was never propagated. The three `elev_quality` vocabularies also
disagree with each other (0301 is "Surveyed NAD83" in the compiled sheet, "2019 RTN GNSS CGVD2013"
in EPA and AGS).

**Coordinates.** `GOWN_EASTING`/`GOWN_NORTHING`, copied into the AGS master as `easting`/`northing`,
do not match their own lat/lon: easting agrees with NAD83 UTM 11N to ±7 m but **northing is ~220 m
short**, matching NAD27 instead. It is a mixed, incoherent pair. **0931 Evan-Thomas Creek is
4,667 m due south** of where both its lat/lon and its own legal description put it. `dev_m` below
carries the reprojection residual so the error is visible as data rather than prose.

In [43]:
# (quantity, source_label, frame, column, vintage_column_or_None)
_ELEV_SPEC = [
    ("top_of_casing", "wiski",      W, "GWREF_DATUM",                             None),
    ("top_of_casing", "xlx04",      S, "Well Elevation (m)",                      None),
    ("top_of_casing", "compiled",   K, "stngroundwaterdatum",                     "stnelevationquality"),
    ("top_of_casing", "ags_2019",   A, "toc_elev_masl",                           "elev_quality"),
    ("top_of_casing", "epa_2019",   E, "GOWN_WISKI_TopOfCasingElev",              "GOWN_ElevationQuality"),
    ("top_of_casing", "epa_2012",   E, "GOWN_TopOfCasingElev_July9_2012",         None),
    ("top_of_casing", "epa_pre2012", E, "GOWN_TopOfCasingElev_Pre_July2012",      None),
    ("top_of_casing", "xs_collars", X, "mp_elev_m",                               None),
    ("top_of_casing", "awwid",      X, "awwid_elev_m",                            None),
    ("ground",        "dem25m",     X, "dem_ground_m",                            None),
    ("ground",        "gfav3",      A, "ground_elev_gfav3_masl",                  None),
    ("ground",        "gfav3_epa",  E, "GOWN_GroundElevGFAv3_masl",               None),
    ("pipe_height",   "epa",        E, "GOWN_GWPipeHeight",                       None),
    ("pipe_height",   "compiled",   K, "stngroundwaterpipeheight",                None),
    ("completion_top", "gfav3",     A, "compl_top_elev_masl",                     None),
    ("completion_top", "gfav3_epa", E, "GOWN_TopCompletionElevGFAv3_masl",        None),
    ("completion_top", "gfav3_cmp", K, "Top Completion Elevation using GFAv3 (masl)", None),
    ("completion_mid", "gfav3",     A, "compl_mid_elev_masl",                     None),
    ("completion_mid", "gfav3_epa", E, "GOWN_MidCompletionElevGFAv3_masl",        None),
    ("completion_mid", "gfav3_cmp", K, "Mid Completion Elevation using GFAv3 (masl)", None),
    ("completion_bot", "gfav3",     A, "compl_bot_elev_masl",                     None),
    ("completion_bot", "gfav3_epa", E, "GOWN_BottomCompletionElevGFAv3_masl",     None),
    ("completion_bot", "gfav3_cmp", K, "Bottom Completion Elevation using GFAv3 (masl)", None),
    ("well_bottom",   "gfav3",      A, "well_bottom_elev_masl",                   None),
    ("well_bottom",   "gfav3_epa",  E, "GOWN_WellBottomCompletionElevGFAv3_masl", None),
    ("well_bottom",   "gfav3_cmp",  K, "Well Bottom Elevation using GFAv3 (masl)", None),
    ("well_bottom",   "xs_collars", X, "mp_base_elev_m",                          None),
]

_rows = []
for quantity, label, frame, col, vintage_col in _ELEV_SPEC:
    vals = pd.to_numeric(RAW[frame][col], errors="coerce").reindex(CODES)
    vint = RAW[frame][vintage_col].reindex(CODES) if vintage_col else pd.Series(None, index=CODES)
    for code in CODES:
        if pd.notna(vals[code]):
            _rows.append({"code": code, "quantity": quantity, "source": label,
                          "value_m": round(float(vals[code]), 4), "vintage": vint[code]})
    note_columns([(frame, col)], "sourced", "well_elevations", quantity)
    if vintage_col:
        note_columns([(frame, vintage_col)], "sourced", "well_elevations", f"{quantity}.vintage")

well_elevations = pd.DataFrame(_rows)
# Two derived columns that only exist as comparisons, not as source values.
note_columns([(X, "elev_discrepancy_m"), (X, "dem_minus_mp_m")], "derived",
             "well_elevations", "spread")
# '1603.125 m' -- the string rendering of the numeric column already captured above.
note_columns([(S, "Well Elevation (mamsl)")], "duplicate", "well_elevations", "top_of_casing")

_spread = (well_elevations.groupby(["code", "quantity"])["value_m"]
           .agg(["min", "max", "count"]).eval("spread = max - min"))
well_elevations = well_elevations.merge(
    _spread["spread"].round(3).rename("quantity_spread_m"), on=["code", "quantity"], how="left")

print(f"well_elevations {well_elevations.shape} | "
      f"{well_elevations.quantity.nunique()} quantities, {well_elevations.source.nunique()} sources")
well_elevations[well_elevations.quantity == "top_of_casing"].pivot(
    index="code", columns="source", values="value_m")

well_elevations (143, 6) | 7 quantities, 14 sources


source,ags_2019,awwid,compiled,epa_2012,epa_2019,epa_pre2012,wiski,xlx04,xs_collars
code,,,,,,,,,
0301,1612.657,1612.66,1603.125,1601.400,1612.657,1601.400,1603.125,1603.125,1603.125
0303,1668.778,1668.78,1668.041,1669.100,1668.778,1669.100,1668.041,1668.041,1668.041
0305,2069.031,2069.03,2067.549,2051.620,2069.031,2051.620,2067.549,2067.549,2067.549
0364,1284.781,1284.78,1284.610,1284.473,1284.781,1284.473,1284.610,1284.610,1284.610
0386,1640.000,NaN,1640.000,1640.000,1640.000,1640.000,1640.000,1640.000,1640.000
0759,1291.838,1291.84,1291.795,1291.669,1291.838,1291.669,1291.795,1291.795,1291.795
0760,1315.345,NaN,1315.463,1315.345,1315.345,1315.345,1315.463,1315.463,1315.463
0764,1380.067,1380.07,1378.093,1377.360,1380.067,1377.360,1378.093,1378.093,1378.093
0931,1510.314,1510.31,1509.849,1509.720,1510.314,1509.720,1509.849,1509.849,1509.849


In [44]:
from pyproj import Transformer

_TO_UTM11N = Transformer.from_crs("EPSG:4326", "EPSG:32611", always_xy=True)
_ref_E, _ref_N = _TO_UTM11N.transform(well_location["longitude"].values,
                                      well_location["latitude"].values)
_ref = pd.DataFrame({"easting": _ref_E, "northing": _ref_N}, index=CODES)

_COORD_SPEC = [
    ("latitude",  "wiski",      W, "station_latitude"),
    ("latitude",  "epa",        E, "GOWN_Latitude"),
    ("latitude",  "ags",        A, "latitude"),
    ("latitude",  "compiled",   K, "stnlatitude"),
    ("latitude",  "xlx04",      S, "Latitude"),
    ("longitude", "wiski",      W, "station_longitude"),
    ("longitude", "epa",        E, "GOWN_Longitude"),
    ("longitude", "ags",        A, "longitude"),
    ("longitude", "compiled",   K, "stnlongitude"),
    ("longitude", "xlx04",      S, "Longitude"),
    ("easting",   "epa",        E, "GOWN_EASTING"),
    ("easting",   "ags",        A, "easting"),
    ("easting",   "compiled",   K, "stneasting"),
    ("easting",   "xs_collars", X, "x_utm11n"),
    ("northing",  "epa",        E, "GOWN_NORTHING"),
    ("northing",  "ags",        A, "northing"),
    ("northing",  "compiled",   K, "stnnorthing"),
    ("northing",  "xs_collars", X, "y_utm11n"),
]

_rows = []
for quantity, label, frame, col in _COORD_SPEC:
    vals = pd.to_numeric(RAW[frame][col], errors="coerce").reindex(CODES)
    for code in CODES:
        if pd.isna(vals[code]):
            continue
        v = float(vals[code])
        if quantity in ("easting", "northing"):
            dev = v - _ref.loc[code, quantity]
        elif quantity == "latitude":
            dev = (v - well_location.loc[code, "latitude"]) * 111_320
        else:
            dev = (v - well_location.loc[code, "longitude"]) * 111_320 * np.cos(np.radians(51))
        _rows.append({"code": code, "quantity": quantity, "source": label,
                      "value": round(v, 6), "dev_m": round(dev, 1)})
    note_columns([(frame, col)], "sourced", "well_coordinates", quantity)

well_coordinates = pd.DataFrame(_rows)
well_coordinates["suspect"] = well_coordinates["dev_m"].abs() > 25

print("Reprojection residual by source (metres), UTM 11N:")
print(well_coordinates[well_coordinates.quantity.isin(["easting", "northing"])]
      .pivot_table(index="source", columns="quantity", values="dev_m",
                   aggfunc=["min", "max"]).round(1).to_string())
print(f"\nsuspect rows: {int(well_coordinates.suspect.sum())}")
well_coordinates[well_coordinates.quantity == "northing"].pivot(
    index="code", columns="source", values="dev_m")

Reprojection residual by source (metres), UTM 11N:
               min              max         
quantity   easting northing easting northing
source                                      
ags            1.3  -4666.1   116.7   -215.5
compiled      -3.9     -0.3     0.6      2.2
epa            1.3  -4666.1   116.7   -215.5
xs_collars     0.0      0.0     0.0      0.0

suspect rows: 20


source,ags,compiled,epa,xs_collars
code,,,,
0301,-222.2,2.2,-222.2,0.0
0303,-217.6,-0.2,-217.6,0.0
0305,-224.6,0.6,-224.6,0.0
0364,-218.5,-0.3,-218.5,0.0
0386,-218.6,NaN,-218.6,0.0
0759,-218.9,0.0,-218.9,0.0
0760,-219.3,NaN,-219.3,0.0
0764,-215.5,-0.0,-215.5,0.0
0931,-4666.1,-0.1,-4666.1,0.0


## 4 · Water levels — 2005–2024 daily

From the wide daily pivots in `data/gown/`, the same source notebooks 01–05 use. The native
sub-hourly record (1964→2026) lives in `data/gown/parquet/active/` and is not used here.

`wl_datum_check` reconstructs the reference datum each reading was reduced to (`absval + mbtoc`),
which is what proves the 2019 re-survey never reached the published elevations — and which turns up
**two datum revisions that no metadata table records**.

In [45]:
def _read_wide(name):
    d = pd.read_csv(GOWN_DIR / name, parse_dates=["date"], index_col="date",
                    usecols=["date"] + CODES)
    return d[CODES]


wl_wide_masl        = _read_wide("combined_absval.csv")
wl_wide_mbtoc       = _read_wide("combined_mbtoc.csv")
wl_wide_masl_filled = _read_wide("combined_absval_20052024_filled.csv")

wl_long = (
    pd.concat({"wl_masl": wl_wide_masl.stack(dropna=False),
               "wl_mbtoc": wl_wide_mbtoc.stack(dropna=False),
               "wl_masl_filled": wl_wide_masl_filled.stack(dropna=False)}, axis=1)
    .rename_axis(["date", "code"]).reset_index()
)
wl_long["is_filled"] = wl_long["wl_masl"].isna() & wl_long["wl_masl_filled"].notna()
wl_long = wl_long.sort_values(["code", "date"]).reset_index(drop=True)


def _longest_gap(s):
    best = run = 0
    for v in s.isna().values:
        run = run + 1 if v else 0
        best = max(best, run)
    return best


wl_coverage = pd.DataFrame({
    "short_name":    ids["short_name"],
    "n_obs":         wl_wide_masl.notna().sum(),
    "n_gap":         wl_wide_masl.isna().sum(),
    "pct_complete":  (wl_wide_masl.notna().mean() * 100).round(1),
    "first_valid":   wl_wide_masl.apply(lambda s: s.first_valid_index()),
    "last_valid":    wl_wide_masl.apply(lambda s: s.last_valid_index()),
    "longest_gap_d": wl_wide_masl.apply(_longest_gap),
    "n_imputed":     wl_wide_masl_filled.notna().sum() - wl_wide_masl.notna().sum(),
    "masl_min":      wl_wide_masl.min().round(3),
    "masl_max":      wl_wide_masl.max().round(3),
    "masl_range":    (wl_wide_masl.max() - wl_wide_masl.min()).round(3),
}).rename_axis("code")

print(f"wl_wide_masl {wl_wide_masl.shape}  "
      f"{wl_wide_masl.index.min():%Y-%m-%d} -> {wl_wide_masl.index.max():%Y-%m-%d}")
print(f"wl_long      {wl_long.shape}   imputed rows: {int(wl_long.is_filled.sum()):,}")
wl_coverage

wl_wide_masl (7305, 9)  2005-01-01 -> 2024-12-31
wl_long      (65745, 6)   imputed rows: 2,691


,short_name,n_obs,n_gap,pct_complete,first_valid,last_valid,longest_gap_d,n_imputed,masl_min,masl_max,masl_range
code,,,,,,,,,,,
0301,Marmot Creek Basin S5250,6778,527,92.8,2005-12-12,2024-12-31,345,527,1600.116,1602.590,2.475
0303,Marmot Creek Basin N5475,6791,514,93.0,2006-05-30,2024-12-31,514,514,1658.769,1666.370,7.601
0305,Marmot Creek Basin N6770,6937,368,95.0,2005-12-12,2024-12-31,345,368,2058.743,2066.969,8.226
0364,Many Springs,7175,130,98.2,2005-01-01,2024-12-31,130,130,1283.269,1284.494,1.225
0386,Marmot Creek Basin N2507E,6849,456,93.8,2005-12-12,2024-12-31,345,456,1633.554,1635.537,1.983
0759,Exshaw,7120,185,97.5,2005-01-01,2024-12-31,155,185,1287.556,1291.854,4.298
0760,Canmore Tourist Info,7135,170,97.7,2005-01-01,2024-12-31,170,170,1309.703,1314.056,4.353
0764,Harvie Heights,7156,149,98.0,2005-01-01,2024-12-31,92,149,1348.573,1357.180,8.607
0931,Evan-Thomas Creek,7113,192,97.4,2005-01-01,2024-12-31,88,192,1496.129,1504.076,7.946


In [46]:
_implied = wl_wide_masl + wl_wide_mbtoc

wl_datum_check = pd.DataFrame({
    "short_name":   ids["short_name"],
    "n":            _implied.notna().sum(),
    "implied_mean": _implied.mean().round(4),
    "implied_std":  _implied.std().round(4),
    "implied_span": (_implied.max() - _implied.min()).round(4),
    "implied_last": _implied.apply(lambda s: s.dropna().iloc[-1]).round(4),
    "wiski_datum":  pd.to_numeric(RAW["meta_wiski"]["GWREF_DATUM"]).reindex(CODES),
    "ags_toc_2019": pd.to_numeric(RAW["meta_ags_master"]["toc_elev_masl"]).reindex(CODES),
}).rename_axis("code")
# Compare the CURRENT datum: for two wells the mean straddles a mid-record step.
wl_datum_check["matches_wiski"] = (
    (wl_datum_check["implied_last"] - wl_datum_check["wiski_datum"]).abs() < 0.01)
wl_datum_check["datum_shifted"] = wl_datum_check["implied_span"] > 0.01

# Feed the reconstructed datum back into well_elevations as its own source.
_implied_rows = [{"code": c, "quantity": "top_of_casing", "source": "implied_from_wl",
                  "value_m": float(wl_datum_check.loc[c, "implied_last"]),
                  "vintage": "reconstructed from absval + mbtoc", "quantity_spread_m": np.nan}
                 for c in CODES]
well_elevations = pd.concat([well_elevations, pd.DataFrame(_implied_rows)], ignore_index=True)

print("Datum revisions inside the 2005-2024 record (recorded nowhere in the metadata):")
for c in wl_datum_check.index[wl_datum_check["datum_shifted"]]:
    s = _implied[c].dropna().round(3)
    ch = s[s != s.shift()]
    print(f"  {c} {ids.loc[c, 'short_name']}: "
          + " -> ".join(f"{v:.3f} m ({d:%Y-%m-%d})" for d, v in ch.items()))
wl_datum_check

Datum revisions inside the 2005-2024 record (recorded nowhere in the metadata):
  0364 Many Springs: 1284.670 m (2005-01-01) -> 1284.645 m (2011-06-02) -> 1284.610 m (2011-06-03)
  0759 Exshaw: 1290.492 m (2005-01-01) -> 1291.795 m (2010-03-25)


,short_name,n,implied_mean,implied_std,implied_span,implied_last,wiski_datum,ags_toc_2019,matches_wiski,datum_shifted
code,,,,,,,,,,
0301,Marmot Creek Basin S5250,6778,1603.1250,0.0001,0.0004,1603.1250,1603.125,1612.657,True,False
0303,Marmot Creek Basin N5475,6791,1668.0410,0.0001,0.0004,1668.0409,1668.041,1668.778,True,False
0305,Marmot Creek Basin N6770,6937,2067.5490,0.0001,0.0005,2067.5489,2067.549,2069.031,True,False
0364,Many Springs,7175,1284.6285,0.0277,0.0603,1284.6100,1284.610,1284.781,True,True
0386,Marmot Creek Basin N2507E,6849,1640.0000,0.0001,0.0004,1640.0000,1640.000,1640.000,True,False
0759,Exshaw,7120,1291.4795,0.5582,1.3033,1291.7950,1291.795,1291.838,True,True
0760,Canmore Tourist Info,7135,1315.4630,0.0001,0.0003,1315.4630,1315.463,1315.345,True,False
0764,Harvie Heights,7156,1378.0930,0.0001,0.0003,1378.0930,1378.093,1380.067,True,False
0931,Evan-Thomas Creek,7113,1509.8490,0.0001,0.0004,1509.8491,1509.849,1510.314,True,False


## 5 · Assessments — the 1996–2008 review history, tidied

Sixteen EPA columns hold free text from successive hydrograph reviews, one column per year per
reviewer. Reshaped to `code · year · kind · assessor · text`.

In [47]:
# (frame, column, year, kind)  -- assessor is parsed out of the text where it is prefixed
_ASSESS_SPEC = [
    (E, "GOWN_F1996_HydrographAssessment", 1996, "hydrograph assessment"),
    (E, "GOWN_F1997_HydrographAssessment", 1997, "hydrograph assessment"),
    (E, "GOWN_F1997_COMMENTS",             1997, "comment"),
    (E, "GOWN_F1997_Recomendation",        1997, "recommendation"),
    (E, "GOWN_F1998_HydrographAssessment", 1998, "hydrograph assessment"),
    (E, "GOWN_F1999_HydrographAssessment", 1999, "hydrograph assessment"),
    (E, "GOWN_F2000_HydrographAssessment", 2000, "hydrograph assessment"),
    (E, "GOWN_F2001_HydrographAssessment", 2001, "hydrograph assessment"),
    (E, "GOWN_F2002_HydrographAssessment", 2002, "hydrograph assessment"),
    (E, "GOWN_F2003_AssessmentAndInterpretation", 2003, "assessment and interpretation"),
    (E, "GOWN_F2008_GISRecommendations",   2008, "recommendation"),
    (E, "GOWN_F2008_GISComments",          2008, "comment"),
    (E, "GOWN_Komex_RECOMMEND",            None, "recommendation"),
    (A, "rec_1997",                        1997, "recommendation"),
    (A, "rec_2008_gis",                    2008, "recommendation"),
    (A, "rec_komex",                       None, "recommendation"),
    (E, "GOWN_StationRemarks",             None, "station remark"),
    (E, "GOWN_Remarks",                    None, "remark"),
]

_rows = []
for frame, col, year, kind in _ASSESS_SPEC:
    s = RAW[frame][col].reindex(CODES)
    src = "epa" if frame == E else "ags"
    for code in CODES:
        v = s[code]
        if pd.isna(v) or not str(v).strip():
            continue
        text = " ".join(str(v).split())
        m = re.match(r"^(?:\d{4}\s*-\s*)?([A-Z][A-Za-z.\s]{2,20}?):\s*(.*)$", text)
        assessor = m.group(1).strip() if m else ("Komex" if "komex" in col.lower() else None)
        _rows.append({"code": code, "year": year, "kind": kind, "assessor": assessor,
                      "source": src, "text": text})
    note_columns([(frame, col)], "reshaped", "well_assessments", kind)

note_columns([(E, "GOWN_AffectedByHumanActivity")], "reshaped", "well_assessments",
             "human activity")
for code, v in RAW[E]["GOWN_AffectedByHumanActivity"].reindex(CODES).items():
    if pd.notna(v):
        _rows.append({"code": code, "year": None, "kind": "affected by human activity",
                      "assessor": None, "source": "epa", "text": str(v)})

well_assessments = (pd.DataFrame(_rows)
                    .drop_duplicates(subset=["code", "year", "kind", "text"])
                    .sort_values(["code", "year", "kind"], na_position="last")
                    .reset_index(drop=True))

print(f"well_assessments {well_assessments.shape} | "
      f"{well_assessments.code.nunique()} wells, kinds: {sorted(well_assessments.kind.unique())}")
well_assessments.head(8)

well_assessments (50, 6) | 9 wells, kinds: ['affected by human activity', 'comment', 'hydrograph assessment', 'recommendation', 'remark', 'station remark']


,code,year,kind,assessor,source,text
0,0301,1996.0,hydrograph assessment,None,epa,"1996 - Cyclical, annual fluctuations of 2m, re..."
1,0301,1997.0,hydrograph assessment,Lorberg,epa,1997 - Lorberg: Relects iniltration and runoff...
2,0301,1997.0,recommendation,None,epa,"1997 - Drop, project completed, leave for HQ"
3,0301,2008.0,recommendation,None,epa,2008 GIS: Reclaim AENV Comments
4,0301,NaN,remark,None,epa,WEPA_No=383
5,0301,NaN,station remark,None,epa,This was an old Research Council Basin study w...
6,0303,1996.0,hydrograph assessment,None,epa,"1996 - Cyclical, annual fluctuations usually n..."
7,0303,1997.0,hydrograph assessment,Lorberg,epa,1997 - Lorberg: Snowmelt has major impact on W...


## 6 · Area tables — township and basin context, not well properties

The EPA sheet carries ~40 `QT1_*` columns of regional water use and pressure, mirrored as ~20
columns in the AGS master with identical numbers (19 exact-duplicate pairs confirmed). They describe
the **township or HUC8**, so they are keyed that way — 5 townships and 2 basins cover all nine wells
— and reached from a well via `well_area_link`.

One exception, caught by the "wells sharing a township must agree" check: despite its
`QT1_2010_GEN_Population_Density_*` naming, **population density is sampled at the well, not the
township**. 0760 (Canmore Tourist Info) reads High / 174.7 per km² while 0764 (Harvie Heights), in
the same township `025-10-W5`, reads Low / 0.46. Those two columns stay on `well_area_link`. The
other six stressors are genuine township aggregates.

In [48]:
# (out_col, epa_column, ags_column_or_None)
_TWP_USE = [
    ("gw_div_2020_m3", "QT1_Combined_GW_Use_TWP_Diversion_Year_2020", None),
    ("gw_div_2021_m3", "QT1_Combined_GW_Use_TWP_Diversion_Year_2021", None),
    ("gw_div_2022_m3", "QT1_Combined_GW_Use_TWP_Diversion_Year_2022", None),
    ("gw_div_2023_m3", "QT1_Combined_GW_Use_TWP_Diversion_Year_2023", None),
    ("gw_div_avg_m3",  "QT1_Combined_GW_Use_TWP_Diversion_Average_Usage_m_3", "gw_div_twp_avg_m3"),
    ("gw_div_max_m3",  "QT1_Combined_GW_Use_TWP_Diversion_Max_One_Year_m_3",  "gw_div_twp_max_m3"),
    ("gw_ret_2020_m3", "QT1_Combined_GW_Use_TWP_Return_Year_2020", None),
    ("gw_ret_2021_m3", "QT1_Combined_GW_Use_TWP_Return_Year_2021", None),
    ("gw_ret_2022_m3", "QT1_Combined_GW_Use_TWP_Return_Year_2022", None),
    ("gw_ret_2023_m3", "QT1_Combined_GW_Use_TWP_Return_Year_2023", None),
    ("gw_ret_avg_m3",  "QT1_Combined_GW_Use_TWP_Return_Average_Usage_m_3", "gw_ret_twp_avg_m3"),
    ("gw_ret_max_m3",  "QT1_Combined_GW_Use_TWP_Return_Max_One_Year_m_3", None),
    ("sw_div_2020_m3", "QT1_Combined_SW_Use_TWP_Diversion_Year_2020", None),
    ("sw_div_2021_m3", "QT1_Combined_SW_Use_TWP_Diversion_Year_2021", None),
    ("sw_div_2022_m3", "QT1_Combined_SW_Use_TWP_Diversion_Year_2022", None),
    ("sw_div_2023_m3", "QT1_Combined_SW_Use_TWP_Diversion_Year_2023", None),
    ("sw_div_avg_m3",  "QT1_Combined_SW_Use_TWP_Diversion_Average_Usage_m_3", "sw_div_twp_avg_m3"),
    ("sw_div_max_m3",  "QT1_Combined_SW_Use_TWP_Diversion_Max_One_Year_m_3",  "sw_div_twp_max_m3"),
    ("sw_ret_2020_m3", "QT1_Combined_SW_Use_TWP_Return_Year_2020", None),
    ("sw_ret_2021_m3", "QT1_Combined_SW_Use_TWP_Return_Year_2021", None),
    ("sw_ret_2022_m3", "QT1_Combined_SW_Use_TWP_Return_Year_2022", None),
    ("sw_ret_2023_m3", "QT1_Combined_SW_Use_TWP_Return_Year_2023", None),
    ("sw_ret_avg_m3",  "QT1_Combined_SW_Use_TWP_Return_Average_Usage_m_3", "sw_ret_twp_avg_m3"),
    ("sw_ret_max_m3",  "QT1_Combined_SW_Use_TWP_Return_Max_One_Year_m_3", None),
]
_HUC_USE = [
    ("gw_div_2020_m3", "QT1_Combined_GW_Use_Huc8_Diversion_Year_2020", "gw_div_huc8_2020_m3"),
    ("gw_div_2021_m3", "QT1_Combined_GW_Use_Huc8_Diversion_Year_2021", "gw_div_huc8_2021_m3"),
    ("gw_div_2022_m3", "QT1_Combined_GW_Use_Huc8_Diversion_Year_2022", "gw_div_huc8_2022_m3"),
    ("gw_div_2023_m3", "QT1_Combined_GW_Use_Huc8_Diversion_Year_2023", "gw_div_huc8_2023_m3"),
    ("gw_div_avg_m3",  "QT1_Combined_GW_Use_Huc8_Diversion_Average_Usage_m_3_", "gw_div_huc8_avg_m3"),
    ("gw_div_max_m3",  "QT1_Combined_GW_Use_Huc8_Diversion_Max_One_Year_m_3_",  "gw_div_huc8_max_m3"),
    ("ems_gw_2023_m3", "QT1_EMS_GW_2000_2023_Huc8_Y_2023", "ems_gw_huc8_2023_m3"),
]
_STRESSORS = [
    ("mining_density_class",  "QT1_2010_GEN_MiningActivity_Density_Class", "mining_density_2010"),
    ("ems_alloc_density_class", "QT1_2010_GW_EMS_Allocation_Density_Class", "ems_alloc_density_2010"),
    ("well_density_class",    "QT1_2010_GW_WaterWell_DistributionDensity_Class", "well_density_2010"),
    ("complaints_class",      "QT1_2010_GW_WaterWell_Complaints_Class", "complaints_2010"),
    ("unlic_domestic_2010_m3", "QT1_2010_GW_Unlic_Domestic_Sum_Volume", "unlic_domestic_2010_m3"),
    ("unlic_domestic_2024_m3", "QT1_2024_GW_Unlic_Domestic_Est1", "unlic_domestic_2024_m3"),
]


def _collapse(spec, key_series, key_name):
    """One row per distinct key; assert every well sharing a key agrees, and record lineage."""
    frame = pd.DataFrame({key_name: key_series})
    for out_col, epa_col, ags_col in spec:
        frame[out_col] = RAW[E][epa_col].reindex(CODES).values
        note_columns([(E, epa_col)], "area", f"area:{key_name}", out_col)
        if ags_col:
            same = _norm_vec(RAW[A][ags_col]) == _norm_vec(RAW[E][epa_col])
            note_columns([(A, ags_col)], "duplicate" if same else "conflict",
                         f"area:{key_name}", out_col)
    g = frame.groupby(key_name).agg(lambda s: s.iloc[0])
    n_unique = frame.groupby(key_name).nunique(dropna=False).max().max()
    assert n_unique <= 1, f"{key_name}: wells sharing a key disagree"
    return g


area_water_use_twp = _collapse(_TWP_USE, well_area_link["twp_id"], "twp_id")
area_water_use_huc8 = _collapse(_HUC_USE, well_area_link["huc8"], "huc8")
area_stressors = _collapse(_STRESSORS, well_area_link["twp_id"], "twp_id")

# Population density is NOT a township aggregate despite its QT1_2010_GEN_* name: it is sampled at
# the well. 0760 (Canmore Tourist Info) reads High / 174.7 per km2 while 0764 (Harvie Heights),
# in the same township 025-10-W5, reads Low / 0.46. So it belongs on the well, not the area table.
well_area_link["site_pop_density_class"] = RAW[E][
    "QT1_2010_GEN_Population_Density_Class"].reindex(CODES)
well_area_link["site_pop_density"] = pd.to_numeric(
    RAW[E]["QT1_2010_GEN_Population_Density_Projection_PopDensity"].reindex(CODES))
note_columns([(E, "QT1_2010_GEN_Population_Density_Class")], "sourced", "well_area_link",
             "site_pop_density_class")
note_columns([(E, "QT1_2010_GEN_Population_Density_Projection_PopDensity")], "sourced",
             "well_area_link", "site_pop_density")
note_columns([(A, "pop_density_2010")], "duplicate", "well_area_link", "site_pop_density_class")
note_columns([(A, "pop_density_proj")], "duplicate", "well_area_link", "site_pop_density")

print(f"area_water_use_twp  {area_water_use_twp.shape}  ({len(area_water_use_twp)} townships)")
print(f"area_water_use_huc8 {area_water_use_huc8.shape}  ({len(area_water_use_huc8)} basins)")
print(f"area_stressors      {area_stressors.shape}  ({len(area_stressors)} townships)")
print("\nsite-level population density (kept on the well, not the township):")
print(well_area_link[["twp_id", "site_pop_density_class", "site_pop_density"]].to_string())
area_water_use_twp[["gw_div_avg_m3", "gw_div_max_m3", "sw_div_avg_m3", "sw_ret_avg_m3"]]

area_water_use_twp  (5, 24)  (5 townships)
area_water_use_huc8 (2, 7)  (2 basins)
area_stressors      (5, 6)  (5 townships)

site-level population density (kept on the well, not the township):
         twp_id site_pop_density_class  site_pop_density
code                                                    
0301  023-09-W5                    Low          0.101883
0303  023-09-W5                    Low          0.101883
0305  023-09-W5                    Low          0.101883
0364  024-08-W5                 Medium          0.456657
0386  023-09-W5                    Low          0.101883
0759  024-09-W5                 Medium          0.456657
0760  025-10-W5                   High        174.743161
0764  025-10-W5                    Low          0.456657
0931  022-09-W5                    Low          0.101883


,gw_div_avg_m3,gw_div_max_m3,sw_div_avg_m3,sw_ret_avg_m3
twp_id,,,,
022-09-W5,258814.0,310410.0,0.0,0.0
023-09-W5,694353.0,694390.0,0.0,0.0
024-08-W5,29121.0,35101.0,114929.0,0.0
024-09-W5,322253.0,585643.0,483227.0,32287.0
025-10-W5,65168.0,65172.0,0.0,0.0


## 7 · Drilling record — completion, lithology, hydraulics, chemistry

From the AWWID extracts plus the derived cross-section tables.

**Units.** AWWID stores depths and elevations in **feet**, diameters and slot sizes in **inches**
(verified: 0759 total depth 720 ft = 219.5 m; screen 640–660 ft = 195.07–201.17 m). Raw columns are
left untouched and metric twins added with an `_m` suffix.

**Traps kept visible rather than silently fixed:**

- `lithologies.Depth` is the interval **base** (cumulative), not a thickness, and rows are **not
  sorted** — 0759 lists 720 ft before 705 ft, 0303 lists 75 and 65 ft after 120 ft.
- Five rows across 0303 and 0305 are `Material == 'Old Well'` placeholders repeating total depth at
  zero thickness — flagged `is_placeholder`, which is why 70 raw rows become 65 intervals.
- `pump_tests.Start_Time` is corrupt (sentinels `1900-01-11`, `2012-09-29` = the DB migration date).
  Use `Test_Date`.
- Chemistry exists for **only 3 wells** (0301, 0303, 0305), all sampled 1965–66.

In [49]:
def _awwid(name, **kw):
    return pd.read_csv(AWWID / f"{name}.csv", low_memory=False, **kw)


AW = {}
AW["wells"] = add_metric(select_wells(_awwid("wells"), "station_no", "station_no", unique=True),
                         ft_cols=["Elevation"])
AW["well_summary"] = add_metric(
    select_wells(_awwid("well_summary"), "station_no", "station_no", unique=True),
    ft_cols=["Elevation", "Total_Depth_Drilled", "Casing_Bottom", "Recommended_Intake_Depth"],
    in_cols=["Casing_OD"])
AW["well_reports"] = add_metric(
    select_wells(_awwid("well_reports"), "station_no", "station_no"),
    ft_cols=["Total_Depth_Drilled", "Finished_Well_Depth", "Casing_Bottom", "Liner_Top",
             "Liner_Bottom", "Annular_Seal_From", "Annular_Seal_To", "Pump_Installed_Depth",
             "Recommended_Intake_Depth", "Distance_Casing_Ground", "Saline_Water_Depth", "Gas_Depth"],
    in_cols=["Casing_OD", "Casing_Thickness", "Liner_OD", "Liner_Thickness", "Screen_Size_OD"])
AW["lithologies"] = add_metric(select_wells(_awwid("lithologies"), "station_no", "station_no"),
                               ft_cols=["Depth"])
AW["lithologies"]["is_placeholder"] = AW["lithologies"]["Material"].eq("Old Well")
AW["screens"] = add_metric(select_wells(_awwid("screens"), "station_no", "station_no"),
                           ft_cols=["From", "To"], in_cols=["Slot_Size"])
AW["perforations"] = add_metric(select_wells(_awwid("perforations"), "station_no", "station_no"),
                                ft_cols=["From", "To"], in_cols=["Diameter"])
AW["boreholes"] = add_metric(select_wells(_awwid("boreholes"), "station_no", "station_no"),
                             ft_cols=["From", "To"], in_cols=["Diameter"])
AW["pump_tests"] = add_metric(select_wells(_awwid("pump_tests"), "station_no", "station_no"),
                              ft_cols=["Static_Water_Level", "End_Water_Level", "Removal_Depth_From"])
AW["pump_test_items"] = add_metric(
    select_wells(_awwid("pump_test_items"), "station_no", "station_no"),
    ft_cols=["Pumping_Depth", "Recovery_Depth"])
AW["chemical_analysis"] = select_wells(_awwid("chemical_analysis"), "station_no", "station_no")
AW["analysis_items"] = select_wells(_awwid("analysis_items"), "station_no", "station_no")
AW["geophysical_logs"] = select_wells(_awwid("geophysical_logs"), "station_no", "station_no")
AW["well_owners"] = select_wells(_awwid("well_owners"), "station_no", "station_no")
AW["drillers"] = _awwid("drillers")
AW["drilling_companies"] = _awwid("drilling_companies")

# --- derived cross-section tables (already metric, sorted, elevation-bearing) --
xs_collars = RAW["xs_collars"]
xs_completions = pd.read_csv(XSEC / "completions.csv")
xs_completions["code"] = xs_completions["code"].astype(str).str.zfill(4)
xs_completions = xs_completions.sort_values(["code", "from_m"]).reset_index(drop=True)
xs_lithology = pd.read_csv(XSEC / "lithology_intervals.csv", index_col=0)
xs_lithology["code"] = xs_lithology["code"].astype(str).str.zfill(4)
xs_lithology = xs_lithology.sort_values(["code", "from_m"]).reset_index(drop=True)

print(f"{len(AW)} AWWID tables")
for k, v in AW.items():
    print(f"   {k:<22} {str(v.shape):>10}")

15 AWWID tables
   wells                     (9, 28)
   well_summary              (9, 38)
   well_reports             (14, 98)
   lithologies              (70, 11)
   screens                   (3, 11)
   perforations              (2, 12)
   boreholes                 (8, 11)
   pump_tests               (22, 17)
   pump_test_items         (283, 10)
   chemical_analysis         (8, 13)
   analysis_items            (98, 8)
   geophysical_logs           (8, 8)
   well_owners               (8, 12)
   drillers                   (3, 7)
   drilling_companies        (7, 12)


In [50]:
# --- completion intervals: the derived metric table is the backbone ----------
well_completion = xs_completions.rename(columns={"type": "interval_type"}).copy()
well_completion["thickness_m"] = (well_completion["to_m"] - well_completion["from_m"]).round(2)
well_completion = well_completion[["code", "interval_type", "source", "from_m", "to_m",
                                   "thickness_m", "top_elev_m", "base_elev_m", "detail"]]

# The GOWN open-interval columns say the same thing; record where they went.
note_columns([(E, "GOWN_Production_m"), (E, "GOWN_TopOfOpenInterval1_m"),
              (E, "GOWN_BottomOfOpenInterval1_m"), (E, "GOWN_TopOfOpenInterval2_m"),
              (E, "GOWN_BottomOfOpenInterval2_m"), (A, "production_interval"),
              (A, "production_interval_epa"), (A, "open1_top_m"), (A, "open1_bot_m"),
              (A, "open2_top_m"), (A, "open2_bot_m"), (K, "stnproductionm"),
              (K, "stnproductionm.1"), (K, "Top of open interval 1 (m)"),
              (K, "bottom of open interval 1 (m)"), (K, "top of open interval 2 (m)"),
              (K, "bottom of open interval 2 (m)"), (S, "Production Interval"),
              (W, "groundwater.Production_S"), (X, "production_str")],
             "sourced", "well_completion", "production interval")

well_lithology_log = xs_lithology.copy()

# --- hydraulics --------------------------------------------------------------
well_hydraulics = AW["pump_tests"].rename(columns={
    "Test_Date": "test_date", "Static_Water_Level_m": "static_wl_m",
    "End_Water_Level_m": "end_wl_m", "Water_Removal_Type": "removal_type",
    "Water_Removal_Rate": "removal_rate", "Removal_Depth_From_m": "removal_depth_m",
    "Reason_for_Short_Test": "note", "Pump_Test_ID": "test_id"}).copy()
well_hydraulics["test_date"] = pd.to_datetime(well_hydraulics["test_date"], errors="coerce")
well_hydraulics["start_time_suspect"] = True     # sentinel values, see the note above
well_hydraulics = well_hydraulics[["code", "test_id", "test_date", "static_wl_m", "end_wl_m",
                                   "removal_type", "removal_rate", "removal_depth_m",
                                   "start_time_suspect", "note"]].sort_values(["code", "test_date"])

well_hydraulic_readings = AW["pump_test_items"].rename(columns={
    "Pump_Test_ID": "test_id", "Minutes": "minutes",
    "Pumping_Depth_m": "pumping_depth_m", "Recovery_Depth_m": "recovery_depth_m"})[
    ["code", "test_id", "minutes", "pumping_depth_m", "recovery_depth_m"]].sort_values(
    ["code", "test_id", "minutes"]).reset_index(drop=True)

# Design/recommendation figures live on the well report, not the test.
_wr = AW["well_reports"]
well_hydraulics_design = _wr.groupby("code").agg(
    recommended_rate=("Recommended_Rate", "first"),
    recommended_intake_depth_m=("Recommended_Intake_Depth_m", "first"),
    artesian_flow_flag=("Artesian_Flow_Flag", "max"),
    artesian_flow_rate=("Artesian_Flow_Rate", "first"),
    pump_installed_flag=("Pump_Installed_Flag", "max"),
    pump_installed_depth_m=("Pump_Installed_Depth_m", "first"),
).reindex(CODES)

# --- water quality -----------------------------------------------------------
well_water_quality = AW["chemical_analysis"].rename(columns={
    "Chemical_Analysis_ID": "analysis_id", "Sample_Date": "sample_date",
    "Analysis_Date": "analysis_date", "Laboratory": "lab", "Remarks": "note"})[
    ["code", "analysis_id", "sample_date", "analysis_date", "lab", "note"]]
well_water_quality["sample_date"] = pd.to_datetime(well_water_quality["sample_date"],
                                                   errors="coerce")
well_wq_analytes = AW["analysis_items"].rename(columns={
    "Chemical_Analysis_ID": "analysis_id", "Element_Name": "analyte",
    "Element_Symbol": "symbol", "Value": "value"})[
    ["code", "analysis_id", "analyte", "symbol", "value"]]
well_wq_wide = (well_wq_analytes
                .pivot_table(index=["code", "analysis_id"], columns="analyte", values="value",
                             aggfunc="first")
                .reset_index()
                .merge(well_water_quality[["code", "analysis_id", "sample_date", "lab"]],
                       on=["code", "analysis_id"], how="left"))
_lead = ["code", "analysis_id", "sample_date", "lab"]
well_wq_wide = well_wq_wide[_lead + [c for c in well_wq_wide.columns if c not in _lead]]
note_columns([(W, "groundwater.1stWQSampleDate_D")], "sourced", "well_water_quality",
             "first_sample_date")
well_water_quality_summary = pd.DataFrame({
    "first_wq_sample_date": RAW[W]["groundwater.1stWQSampleDate_D"].reindex(CODES),
    "n_samples": well_water_quality.groupby("code").size().reindex(CODES).fillna(0).astype(int),
    "n_analytes": well_wq_analytes.groupby("code")["analyte"].nunique().reindex(CODES).fillna(0).astype(int),
}).rename_axis("code")

well_geophysical_logs = AW["geophysical_logs"].rename(columns={
    "Geophysical_Log_ID": "log_id", "Log_Type": "log_type",
    "Log_Taken_Flag": "taken", "Sent_to_AENV_Flag": "sent_to_aenv"})[
    ["code", "log_id", "log_type", "taken", "sent_to_aenv"]]

for _n in ["well_completion", "well_lithology_log", "well_hydraulics",
           "well_hydraulic_readings", "well_hydraulics_design", "well_water_quality",
           "well_wq_analytes", "well_wq_wide", "well_geophysical_logs"]:
    print(f"{_n:<26} {str(eval(_n).shape):>10}")
well_hydraulics.head(6)

well_completion               (14, 9)
well_lithology_log            (65, 9)
well_hydraulics              (22, 10)
well_hydraulic_readings      (283, 5)
well_hydraulics_design         (9, 6)
well_water_quality             (8, 6)
well_wq_analytes              (98, 5)
well_wq_wide                  (8, 22)
well_geophysical_logs          (8, 5)


,code,test_id,test_date,static_wl_m,end_wl_m,removal_type,removal_rate,removal_depth_m,start_time_suspect,note
0,0301,10322264,1966-08-18,1.524000,NaN,Pump,NaN,0.0000,True,NaN
1,0301,10322265,1966-08-18,1.463040,NaN,Pump,NaN,0.0000,True,NaN
2,0301,10322266,1966-08-18,1.463040,NaN,Pump,NaN,0.0000,True,NaN
3,0303,10322260,1989-08-11,6.778752,NaN,Pump,NaN,NaN,True,UNABLE TO GET BY JOINT AT 8' BELOW TOP
6,0303,16009540,1989-09-12,5.117592,NaN,Pump,2.0,32.0040,True,NaN
5,0303,16009539,1990-08-09,4.477512,NaN,Pump,2.0,20.7264,True,NaN


## 8 · Ancillary context — **not** the 9 wells

Surface-water gauges and the neighbouring Lafarge quarry monitoring points. They live in
`data/bow_valley/` but are not GOWN well records, so they keep a separate `ctx_*` namespace.

Two digitising traps are flagged rather than fixed: the `*_approx.csv` files are per-year XY pairs
read off a chart (x = **decimal month**, not a date), and `berm_july_2026.csv`'s first column is
likewise a decimal month despite being headed `Date`.

In [51]:
def read_zrx(path):
    """KISTERS ZRXP export -> DataFrame. Honours RINVAL and the LAYOUT header."""
    head, rows = [], []
    with open(path, encoding="latin-1") as fh:
        for line in fh:
            (head if line.startswith("#") else rows).append(line)
    hdr = "".join(head)
    rinval = float(m.group(1)) if (m := re.search(r"RINVAL(-?[\d.]+)", hdr)) else -777.0
    layout = m.group(1).split(",") if (m := re.search(r"LAYOUT\(([^)]*)\)", hdr)) else \
        ["timestamp", "value", "status"]
    meta = {k: (m.group(1).strip() if (m := re.search(rf"{k}([^|]*)\|", hdr)) else None)
            for k in ("SANR", "SNAME", "SWATER", "TZ", "CUNIT")}
    d = pd.DataFrame([ln.split() for ln in rows if ln.strip()], columns=layout)
    d["timestamp"] = pd.to_datetime(d["timestamp"], format="%Y%m%d%H%M%S")
    d["value"] = pd.to_numeric(d["value"], errors="coerce").replace(rinval, np.nan)
    d = d.set_index("timestamp")
    d.attrs.update(meta)
    return d


def read_wsc_realtime(path):
    """WSC real-time export: 9 preamble lines, then Date (MST), Parameter, Value."""
    d = pd.read_csv(path, skiprows=9)
    d.columns = [c.strip() for c in d.columns]
    d = d.rename(columns={d.columns[0]: "datetime", "Parameter": "parameter"})
    d["datetime"] = pd.to_datetime(d["datetime"])
    return d.set_index("datetime")


def read_approx(path):
    """Digitised chart traces: two header rows (year, then X/Y) -> tidy long."""
    raw = pd.read_csv(path, header=None, dtype=str)
    years = raw.iloc[0].ffill()
    data = raw.iloc[2:].reset_index(drop=True)
    out = []
    for i in range(0, data.shape[1] - 1, 2):
        if pd.isna(years.iloc[i]):
            continue
        sub = data.iloc[:, [i, i + 1]].apply(pd.to_numeric, errors="coerce").dropna()
        sub.columns = ["month_decimal", "elevation"]
        sub.insert(0, "year", int(float(years.iloc[i])))
        out.append(sub)
    return pd.concat(out, ignore_index=True).sort_values(
        ["year", "month_decimal"]).reset_index(drop=True)

In [52]:
# --- Bow River at Banff (WSC 05BB001) -----------------------------------------
_hist = pd.read_csv(STREAM / "05BB001_historical.csv", skiprows=1)
_hist.columns = [c.strip() for c in _hist.columns]
_hist["Date"] = pd.to_datetime(_hist["Date"], format="%Y/%m/%d", errors="coerce")
ctx_bow_banff_daily = (_hist.dropna(subset=["Date"])
                       .pivot_table(index="Date", columns="PARAM", values="Value", aggfunc="first")
                       .rename(columns={1: "discharge_cms", 2: "stage_m"}).rename_axis("date"))
ctx_bow_banff_rt = {
    "2026_daily": read_wsc_realtime(STREAM / "05BB001_2026.csv"),
    "2026_hr":    read_wsc_realtime(STREAM / "05BB001_2026_hr.csv"),
    "hr":         read_wsc_realtime(STREAM / "05BB001_hr.csv"),
    "june":       read_wsc_realtime(STREAM / "brab_june.csv"),
}
ctx_bow_banff_elev = pd.read_csv(STREAM / "brab_elevation.csv", parse_dates=["Date"],
                                 index_col="Date")

# --- Bow River at Canmore ------------------------------------------------------
ctx_bow_canmore_elev = pd.read_csv(STREAM / "cr.csv", parse_dates=["Date"], index_col="Date")
ctx_bow_canmore_flow = pd.read_csv(STREAM / "cr_flow.csv", index_col=0,
                                   parse_dates=["Date"]).set_index("Date")

# --- Kananaskis River near Seebe / Barrier Lake -------------------------------
ctx_kananaskis_seebe_zrx     = read_zrx(STREAM / "05BF001_KanRivSeebe_Day.Mean.zrx")
ctx_kananaskis_seebe_monthly = read_zrx(STREAM / "05BF001_KanRivSeebe_Month.Mean.Prelim.zrx")
ctx_kananaskis_seebe_daily   = pd.read_csv(STREAM / "krns_daily.csv", parse_dates=["date"],
                                           index_col="date")
ctx_kananaskis_seebe_mon_csv = pd.read_csv(STREAM / "krns.csv", parse_dates=["Date"],
                                           index_col="Date")
ctx_barrier_lake_daily       = read_zrx(STREAM / "05BF025_BarrierLk_TAU_Day.Mean.zrx")

# --- Lafarge quarry monitoring points (adjacent to Exshaw 0759) ----------------
ctx_lafarge_lagoon           = pd.read_csv(STREAM / "lagoon_combined.csv", parse_dates=["Date"])
ctx_lafarge_lagoon_exact     = pd.read_csv(STREAM / "lagoon.csv", parse_dates=["Date"],
                                           index_col="Date")
ctx_lafarge_lagoon_digitised = read_approx(STREAM / "lagoon_approx.csv")
ctx_lafarge_berm             = pd.read_csv(BIGHORN / "berm_combined.csv", parse_dates=["Date"])
ctx_lafarge_berm_exact       = pd.read_csv(BIGHORN / "berm.csv", parse_dates=["Date"],
                                           index_col="Date")
ctx_lafarge_berm_digitised   = read_approx(BIGHORN / "berm_approx.csv")
ctx_lafarge_berm_2026 = (pd.read_csv(BIGHORN / "berm_july_2026.csv")
                         .rename(columns={"Date": "month_decimal", "Elevation": "elevation"}))
ctx_lafarge_berm_2026["month_decimal"] = pd.to_numeric(ctx_lafarge_berm_2026["month_decimal"],
                                                       errors="coerce")

_fw = pd.ExcelFile(STREAM / "MD GW data w lafarge overlay (Fleetwood).xlsx")


def _read_fleetwood(sheet):
    d = _fw.parse(sheet, header=None).iloc[3:, 1:6]
    d.columns = ["date", "ref_elevation_m", "depth_m", "elevation_m", "month_decimal"]
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    return d.dropna(subset=["date"]).reset_index(drop=True)


ctx_fleetwood_lagoon = _read_fleetwood("Lagoon Well")
ctx_fleetwood_berm   = _read_fleetwood("Berm Irrigation Well")

print(f"ctx_bow_banff_daily      {ctx_bow_banff_daily.shape}  "
      f"{ctx_bow_banff_daily.index.min():%Y-%m-%d} -> {ctx_bow_banff_daily.index.max():%Y-%m-%d}")
print(f"ctx_kananaskis_seebe_zrx {ctx_kananaskis_seebe_zrx.shape}  "
      f"{ctx_kananaskis_seebe_zrx.attrs['SNAME']}")
print(f"ctx_barrier_lake_daily   {ctx_barrier_lake_daily.shape}  "
      f"{ctx_barrier_lake_daily.attrs['SNAME']}")

ctx_bow_banff_daily      (42408, 2)  1909-05-25 -> 2025-12-31
ctx_kananaskis_seebe_zrx (37978, 2)  Kananaskis River near Seebe
ctx_barrier_lake_daily   (10019, 2)  Barrier Lake near Seebe - TAU (Q from 05BF025)


## 9 · `column_lineage` — where every source column went

The sweep below claims every column not already accounted for by a theme, so the lineage table
covers all 443. Statuses: `primary` / `duplicate` / `fallback` / `conflict` from the theme builder,
plus `sourced` (kept in the elevation, coordinate or completion tables), `reshaped` (tidied into a
long table), `area`, `all_null` and `dropped`.

In [53]:
# Anything still unclaimed is either empty for these 9 wells or a harvest artifact.
_DROP_REASON = {
    "creationDateInMillis": "API harvest timestamp",
    "_links": "API navigation links",
    "station_elevation": "constant 0.0 placeholder, not an elevation",
    "Is it in new formation format?": "AGS worksheet scratch flag",
    "Is this in new formation format?": "AGS worksheet scratch flag",
    "Converted to formation": "AGS worksheet scratch column",
    "AGS Updated": "AGS worksheet scratch column",
}

_claimed = {(r["source_frame"], r["source_column"]) for r in LINEAGE}
for frame_name, d in RAW.items():
    for col in d.columns:
        if (frame_name, col) in _claimed:
            continue
        n = int(d[col].notna().sum())
        if n == 0:
            note_columns([(frame_name, col)], "all_null")
        else:
            note_columns([(frame_name, col)], "dropped", "",
                         _DROP_REASON.get(str(col), "unclaimed"))

column_lineage = pd.DataFrame(LINEAGE)[
    ["source_frame", "source_column", "n_populated", "theme", "target_column", "status"]]

_total = sum(len(d.columns) for d in RAW.values())
_covered = column_lineage.groupby(["source_frame", "source_column"]).ngroups
print(f"source columns: {_total} | covered by lineage: {_covered} "
      f"({len(column_lineage)} rows -- a few columns feed more than one target)")
print(f"\nby status:\n{column_lineage['status'].value_counts().to_string()}")
print(f"\nredundancy: {(column_lineage.status == 'duplicate').sum()} columns are exact copies "
      f"of a value already held elsewhere")
_unclaimed = column_lineage[column_lineage.target_column == "unclaimed"]
if len(_unclaimed):
    print(f"\n!! {len(_unclaimed)} columns fell through:")
    print(_unclaimed[["source_frame", "source_column", "n_populated"]].to_string(index=False))
column_lineage.head(12)

source columns: 443 | covered by lineage: 443 (455 rows -- a few columns feed more than one target)

by status:
status
duplicate    186
primary      109
sourced       71
area          37
all_null      21
reshaped      19
dropped        5
conflict       4
derived        2
fallback       1

redundancy: 186 columns are exact copies of a value already held elsewhere


,source_frame,source_column,n_populated,theme,target_column,status
0,meta_wiski,station_no,9,well_identity,station_no,primary
1,meta_wiski_stations,station_no,9,well_identity,station_no,duplicate
2,meta_epa_intersections,GOWN_WISKI_StationNumber,9,well_identity,station_no,duplicate
3,meta_ags_master,wiski_station_no,9,well_identity,station_no,duplicate
4,meta_compiled,stnnumber,9,well_identity,station_no,duplicate
5,meta_station_info,Station Number,9,well_identity,station_no,duplicate
6,xs_collars,station_no,9,well_identity,station_no,duplicate
7,meta_wiski,groundwater.StationCode_S,9,well_identity,station_code,primary
8,meta_wiski_stations,gown,9,well_identity,station_code,duplicate
9,meta_epa_intersections,GOWN_NO,9,well_identity,station_code,duplicate


In [54]:
# Everything still tagged `conflict` after well_elevations and well_coordinates took the
# elevation and coordinate cases. Expanded to the actual disagreeing values.
_rows = []
for _, r in column_lineage[column_lineage.status == "conflict"].iterrows():
    tgt = eval(r.theme)[r.target_column]
    src = RAW[r.source_frame][r.source_column].reindex(CODES)
    for code in CODES:
        a, b = tgt[code], src[code]
        if pd.isna(b) or str(a).strip().casefold() == str(b).strip().casefold():
            continue
        _rows.append({"code": code, "field": r.target_column, "kept": a,
                      "source_frame": r.source_frame, "source_value": b})

meta_conflicts = pd.DataFrame(_rows).drop_duplicates()
print(f"{len(meta_conflicts)} residual disagreements across "
      f"{meta_conflicts['field'].nunique()} fields "
      f"(elevations and coordinates are handled in their own tables)\n")
print(meta_conflicts.to_string(index=False))
print("""
All three are vocabulary granularity rather than contradiction:
  station_name  0931  'Evans-Thomas Creek DKB' is the AGS/EPA long form of 'Evan-Thomas Creek'
  status        the compiled sheet records a coarse 'Active' where AGS splits WL from WQ
  owner         the compiled sheet names the department (EPA); AGS/EPA name the branch (EMSD/AEP)
""")

13 residual disagreements across 3 fields (elevations and coordinates are handled in their own tables)

code        field                   kept           source_frame                source_value
0931 station_name Evan-Thomas Creek_0931 meta_epa_intersections Evans-Thomas Creek DKB_0931
0931 station_name Evan-Thomas Creek_0931        meta_ags_master Evans-Thomas Creek DKB_0931
0760       status         Active WL & WQ          meta_compiled                      Active
0931       status         Active WL only          meta_compiled                      Active
0301        owner                   EMSD          meta_compiled                         EPA
0303        owner                   EMSD          meta_compiled                         EPA
0305        owner                   EMSD          meta_compiled                         EPA
0364        owner                   EMSD          meta_compiled                         EPA
0386        owner                   EMSD          meta_compiled     

## 10 · Registry and inventory

In [55]:
_CATALOGUE = [
    ("ids",                     "well",  "identity crosswalk; 0764 GIC != Well_ID"),
    ("well_identity",           "well",  "names, aliases, every ID system"),
    ("well_location",           "well",  "verified coords, legal survey, county"),
    ("well_coordinates",        "well",  "sourced; dev_m = reprojection residual"),
    ("well_elevations",         "well",  "sourced; up to 5 TOC values per well"),
    ("well_construction",       "well",  "drilling, casing, diameters, depths"),
    ("well_completion",         "well",  "screens / perforations / production intervals"),
    ("well_geology",            "well",  "aquifer, formation, lithology, HSU, surficial"),
    ("well_lithology_log",      "well",  "driller's log, metric with elevations"),
    ("well_hydraulics",         "well",  "one row per pump test"),
    ("well_hydraulic_readings", "well",  "drawdown / recovery readings"),
    ("well_hydraulics_design",  "well",  "recommended rate, artesian, pump installed"),
    ("well_monitoring",         "well",  "network, status, streams, ownership, use"),
    ("well_assessments",        "well",  "1996-2008 reviews, tidied"),
    ("well_water_quality",      "well",  "chemistry samples"),
    ("well_wq_analytes",        "well",  "analyte values, long"),
    ("well_wq_wide",            "well",  "sample x analyte"),
    ("well_water_quality_summary", "well", "per-well chemistry availability"),
    ("well_geophysical_logs",   "well",  "which logs exist"),
    ("well_area_link",          "well",  "code -> twp_id / huc8"),
    ("area_water_use_twp",      "area",  "township diversion / return volumes"),
    ("area_water_use_huc8",     "area",  "basin diversion volumes"),
    ("area_stressors",          "area",  "mining / well density, unlicensed domestic use"),
    ("wl_wide_masl",            "wl",    "2005-2024 daily, gaps preserved"),
    ("wl_wide_mbtoc",           "wl",    "metres below top of casing"),
    ("wl_wide_masl_filled",     "wl",    "gap-filled"),
    ("wl_long",                 "wl",    "tidy, with is_filled flag"),
    ("wl_coverage",             "wl",    "per-well completeness"),
    ("wl_datum_check",          "wl",    "QC: reconstructed reference datum"),
    ("column_lineage",          "meta",  "where all 443 source columns went"),
    ("meta_conflicts",          "meta",  "residual disagreements outside elev / coords"),
    ("ref_ab_formations",       "ref",   "stratigraphic lookup"),
    ("ref_ags_formations",      "ref",   "stratigraphic lookup"),
    ("ref_layer_legend",        "ref",   "legend for the intersected GIS layers"),
    ("xs_collars",              "xs",    "cross-section collars"),
    ("xs_completions",          "xs",    "cross-section completions"),
    ("xs_lithology",            "xs",    "cross-section lithology"),
    ("ctx_bow_banff_daily",     "ctx",   "WSC 05BB001, 1909-"),
    ("ctx_bow_banff_elev",      "ctx",   ""),
    ("ctx_bow_canmore_elev",    "ctx",   ""),
    ("ctx_bow_canmore_flow",    "ctx",   ""),
    ("ctx_kananaskis_seebe_zrx", "ctx",  "ZRXP daily mean, 1912-"),
    ("ctx_kananaskis_seebe_monthly", "ctx", "ZRXP monthly, prelim"),
    ("ctx_kananaskis_seebe_daily",   "ctx", "krns_daily.csv"),
    ("ctx_kananaskis_seebe_mon_csv", "ctx", "krns.csv, monthly"),
    ("ctx_barrier_lake_daily",  "ctx",   "SANR 05BF024, Q from 05BF025"),
    ("ctx_lafarge_lagoon",      "ctx",   "exact + digitised combined"),
    ("ctx_lafarge_lagoon_exact", "ctx",  ""),
    ("ctx_lafarge_lagoon_digitised", "ctx", "x = decimal month"),
    ("ctx_lafarge_berm",        "ctx",   "exact + digitised combined"),
    ("ctx_lafarge_berm_exact",  "ctx",   ""),
    ("ctx_lafarge_berm_digitised", "ctx", "x = decimal month"),
    ("ctx_lafarge_berm_2026",   "ctx",   "x = decimal month despite 'Date' header"),
    ("ctx_fleetwood_lagoon",    "ctx",   "Fleetwood xlsx, headerless"),
    ("ctx_fleetwood_berm",      "ctx",   "Fleetwood xlsx, headerless"),
]

DATA = {name: eval(name) for name, *_ in _CATALOGUE}
DATA["ctx_bow_banff_rt"] = ctx_bow_banff_rt
DATA["RAW"] = RAW
DATA["AWWID"] = AW


def _period(df):
    idx = df.index if isinstance(df.index, pd.DatetimeIndex) else None
    if idx is None:
        for c in ("date", "Date", "datetime", "sample_date", "test_date", "drill_date"):
            if c in df.columns:
                idx = pd.to_datetime(df[c], errors="coerce").dropna()
                break
    return "" if idx is None or len(idx) == 0 else f"{idx.min():%Y-%m-%d} -> {idx.max():%Y-%m-%d}"


def _n_wells(df):
    if df.index.name == "code":
        return df.index.nunique()
    return df["code"].nunique() if "code" in df.columns else np.nan


inventory = pd.DataFrame([
    {"name": n, "group": g, "rows": len(DATA[n]), "cols": DATA[n].shape[1],
     "wells": _n_wells(DATA[n]), "period": _period(DATA[n]), "note": note}
    for n, g, note in _CATALOGUE])

print(f"{len(_CATALOGUE)} themed tables + RAW ({len(RAW)} frames) + AWWID ({len(AW)} frames)")
print(f"{inventory['rows'].sum():,} rows total\n")
inventory

55 themed tables + RAW (9 frames) + AWWID (15 frames)
214,103 rows total



,name,group,rows,cols,wells,period,note
0,ids,well,9,6,9.0,,identity crosswalk; 0764 GIC != Well_ID
1,well_identity,well,9,17,9.0,,"names, aliases, every ID system"
2,well_location,well,9,17,9.0,,"verified coords, legal survey, county"
3,well_coordinates,well,158,6,9.0,,sourced; dev_m = reprojection residual
4,well_elevations,well,152,6,9.0,,sourced; up to 5 TOC values per well
5,well_construction,well,9,11,9.0,1964-10-01 -> 2001-10-01,"drilling, casing, diameters, depths"
6,well_completion,well,14,9,9.0,,screens / perforations / production intervals
7,well_geology,well,9,24,9.0,,"aquifer, formation, lithology, HSU, surficial"
8,well_lithology_log,well,65,9,9.0,,"driller's log, metric with elevations"
9,well_hydraulics,well,22,10,7.0,1966-08-18 -> 2001-10-31,one row per pump test


## 11 · Summary

In [56]:
print("=" * 104)
print("WHERE THE 443 SOURCE COLUMNS WENT")
print("=" * 104)
print(pd.crosstab(column_lineage["theme"], column_lineage["status"]).to_string())

print("\n" + "=" * 104)
print("TOP-OF-CASING ELEVATION BY SOURCE  (m)")
print("=" * 104)
_toc = well_elevations[well_elevations.quantity == "top_of_casing"]
print(_toc.pivot_table(index="code", columns="source", values="value_m", aggfunc="first")
      .join(ids["short_name"]).to_string())

print("\n" + "=" * 104)
print("COORDINATE ERROR BY SOURCE  (metres from the reprojected lat/lon)")
print("=" * 104)
print(well_coordinates[well_coordinates.quantity.isin(["easting", "northing"])]
      .pivot_table(index="code", columns=["quantity", "source"], values="dev_m", aggfunc="first")
      .to_string())

print("\n" + "=" * 104)
print("WATER-LEVEL COVERAGE  (2005-2024 daily)")
print("=" * 104)
print(wl_coverage.to_string())

print("\n" + "=" * 104)
print("REFERENCE-DATUM QC")
print("=" * 104)
print(wl_datum_check.to_string())

WHERE THE 443 SOURCE COLUMNS WENT
status              all_null  area  conflict  derived  dropped  duplicate  fallback  primary  reshaped  sourced
theme                                                                                                          
                          21     0         0        0        5          0         0        0         0        0
area:huc8                  0     7         0        0        0          7         0        0         0        0
area:twp_id                0    30         0        0        0         12         0        0         0        0
well_area_link             0     0         0        0        0         19         0        9         0        2
well_assessments           0     0         0        0        0          0         0        0        19        0
well_completion            0     0         0        0        0          0         0        0         0       20
well_construction          0     0         0        0        0        

| table | grain | subject |
|---|---|---|
| `well_identity` | well | names, aliases, all the ID systems |
| `well_location` | well | verified coordinates, legal survey, county |
| `well_coordinates` | long | every candidate coordinate **with its source and error** |
| `well_elevations` | long | every candidate elevation **with its source and survey vintage** |
| `well_construction` | well | drilling, casing, liner, seal, diameters, depths |
| `well_completion` | interval | screens, perforations, open/production intervals |
| `well_geology` | well | aquifer, formation, lithology, HSU, surficial + bedrock geology |
| `well_lithology_log` | interval | the driller's depth log, metric with elevations |
| `well_hydraulics` | test | pump tests: static level, rate, drawdown summary |
| `well_hydraulic_readings` | reading | drawdown / recovery time series |
| `well_monitoring` | well | network, status, data streams, ownership, well use |
| `well_assessments` | statement | 1996–2008 hydrograph reviews and recommendations |
| `well_water_quality` / `well_wq_analytes` | sample / analyte | chemistry |
| `well_geophysical_logs` | log | which logs exist |
| `well_area_link` | well | `code → twp_id, huc8` — joins the area tables |
| `area_water_use` / `area_stressors` | township / basin | regional water use and pressures |
| `wl_*` | day | water levels, 2005–2024 daily |
| `ctx_*` | varies | surface water and Lafarge points — **not** these wells |

In [68]:
well_hydraulic_readings

,code,test_id,minutes,pumping_depth_m,recovery_depth_m
0,0301,10322264,0.0,1.530096,NaN
1,0301,10322264,1.0,1.551432,NaN
2,0301,10322264,2.0,1.557528,NaN
3,0301,10322264,4.0,1.554480,NaN
4,0301,10322264,5.0,1.560576,NaN
...,...,...,...,...,...
278,0760,10390957,5.0,NaN,7.6200
279,0760,10390957,6.0,NaN,5.1816
280,0760,10390957,7.0,NaN,3.6576
281,0760,10390957,8.0,NaN,3.0480
